# Modeling Island Effects on Swell
This notebook breaks down three approaches to account for islands affecting incoming swell, outlines their technical debt, and implements the **Line‑of‑Sight Shadowing** method as a practical solo‑developer solution.

---

## 1️⃣ Line‑of‑Sight Shadowing
**Concept:** Cast straight rays from the buoy toward the surf spot. If a ray intersects an island polygon, energy from that direction is blocked.

**Technical Debt:**
- **Ignores diffraction** (no bending around edges).
- **Ignores bathymetry effects** (refraction/shoaling).
- **Simplistic**, may misclassify partially blocked swells.

**Use Case:** Solo developer rapid prototype—easy GIS implementation, fast computation.

In [ ]:
import math
from shapely.geometry import Point, LineString
import geopandas as gpd
from pyproj import Transformer

# Example setup (replace with real paths/coordinates)
islands = gpd.read_file('islands_coastline.shp').to_crs('EPSG:3857')
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
buoy_lon, buoy_lat = -157.5, 21.5  # Example coords
spot_lon, spot_lat = -157.7, 21.6
bx, by = transformer.transform(buoy_lon, buoy_lat)
sx, sy = transformer.transform(spot_lon, spot_lat)
buoy_pt = Point(bx, by)

def is_blocked(theta_deg):
    dx = math.cos(math.radians(theta_deg))
    dy = math.sin(math.radians(theta_deg))
    ray = LineString([buoy_pt, (bx + dx * 1e6, by + dy * 1e6)])
    return islands.intersects(ray).any()

# Test some directions
for θ in [0, 45, 90, 180, 270]:
    print(f"Direction {θ}° blocked? {is_blocked(θ)}")

## 2️⃣ First‑Order Diffraction
**Concept:** Apply Fresnel diffraction around island tips so some energy skirts edges.

**Technical Debt:**
- **Approximate** Fresnel integrals—only first‑order effect.
- Requires island tip identification and wavelength estimates.
- More complex math—maintenance overhead.

**When to Use:** When partial bending materially affects surf; medium complexity.

## 3️⃣ Full Ray‑Tracing & Refraction
**Concept:** Run a wave model (e.g., SWAN) over bathymetry; solves mild‑slope or shallow‑water wave equations.

**Technical Debt:**
- **High setup cost:** bathymetry grids, boundary conditions.
- **Compute‑intensive:** HPC or powerful workstations required.
- **Complexity:** steep learning curve and specialized tools.

**When to Use:** Production forecasting with high accuracy demands.

## 🎯 Recommended Approach for Solo Developers
Use **Line‑of‑Sight Shadowing** as a first iteration:
- Quick to implement with Python + Shapely
- Low dependencies and compute needs
- Refine later with diffraction or full models if needed

---

## 🤔 Does Surfline’s Lotus Model Use Ray‑Tracing/Refraction?
Surfline’s proprietary **Lotus** model is a high‑resolution wave forecasting system that incorporates physics‑based wave propagation, including **refraction**, **shoaling**, and **diffraction** on complex bathymetry (similar in scope to SWAN). While they don’t publicly disclose every detail, available documentation and interviews suggest that Lotus uses a full wave model—so yes, it effectively performs ray‑tracing and refraction simulations rather than a simple shadowing approach.